### LLM Inference | Annotated Sample
Run a selected prompt against the annotated dataset and save summary accuracy JSON.

### Setup

In [ ]:
%pip install pandas openpyxl requests tqdm

In [ ]:
# Import required libraries
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display
from tqdm.auto import tqdm

### Define user inputs

In [ ]:
# Define user inputs
MODEL_NAME = "qwen/qwen3-32b"
API_KEY = os.getenv("GROQ_API_KEY", "").strip()
OUTPUT_NAME = "qwen_qwen3-32b_2"
PROMPT_SELECTION = "prompt_1" 

if not API_KEY:
    print("GROQ_API_KEY is not set. Set it in your environment before running inference.")

# Define input and output paths
FILE_PATH = Path("data/annotated_emotions.xlsx")
PROMPTS_DIR = Path("prompts")
RESULTS_DIR = Path("llm_results")
TEXT_COL = "text"
EXPECTED_COL = "expected_emotion"
PREDICT_COL = "llm_predict"

# Define API settings
API_BASE_URL = "https://api.groq.com/openai/v1"
API_URL = f"{API_BASE_URL}/chat/completions"
REQUEST_GAP_S = 1.0
REQUEST_TIMEOUT_S = 60
MAX_RETRIES = 5
BACKOFF_BASE = 2.0

EMOTION_ORDER = [
    "FRUSTRATED",
    "HESITANT",
    "NEUTRAL",
    "INTERESTED",
    "SATISFIED",
    "EXCITED",
]

### Load the selected prompt

In [ ]:
# Resolve the selected prompt file
selection = str(PROMPT_SELECTION).strip()
if selection.isdigit():
    selection = f"prompt_{selection}"

PROMPT_PATH = PROMPTS_DIR / selection
if not PROMPT_PATH.exists():
    raise FileNotFoundError(f"Prompt file not found: {PROMPT_PATH}")

PROMPT_TEMPLATE = PROMPT_PATH.read_text(encoding="utf-8").strip()
if not PROMPT_TEMPLATE:
    raise ValueError(f"Prompt file is empty: {PROMPT_PATH}")

print(f"Using prompt file: {PROMPT_PATH}")
print(f"Prompt length: {len(PROMPT_TEMPLATE)} characters")

### Load the annotated dataset

In [ ]:
# Load the annotated dataset
df = pd.read_excel(FILE_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

prep_df = df[[TEXT_COL, EXPECTED_COL]].copy()
prep_df[TEXT_COL] = prep_df[TEXT_COL].astype(str).str.strip()
prep_df[EXPECTED_COL] = prep_df[EXPECTED_COL].astype(str).str.strip().str.upper()
display(prep_df.head())

### Define helper functions

In [ ]:
# Helper function to build chat messages for each text sample
def build_messages(prompt_template: str, text: str) -> list[dict[str, str]]:
    if "{text}" in prompt_template:
        return [{"role": "user", "content": prompt_template.format(text=text)}]

    return [
        {"role": "system", "content": prompt_template},
        {"role": "user", "content": text},
    ]

# Helper function to normalize model output into one valid emotion label
def extract_label(raw_text: str | None) -> str | None:
    if raw_text is None:
        return None

    cleaned = raw_text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`").strip()
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:].strip()

    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            for key in ["emotion", "label", "prediction", "predicted_emotion", "class"]:
                if key in parsed and parsed[key] is not None:
                    cleaned = str(parsed[key]).strip()
                    break
    except json.JSONDecodeError:
        pass

    upper = cleaned.upper()
    if upper in EMOTION_ORDER:
        return upper

    matches = [emotion for emotion in EMOTION_ORDER if re.search(rf"\\b{re.escape(emotion)}\\b", upper)]
    if len(matches) == 1:
        return matches[0]

    return None

# Helper function to call the chat completion API with retry and backoff
def call_llm(text: str) -> str | None:
    messages = build_messages(PROMPT_TEMPLATE, text)
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": 0,
    }

    for attempt in range(MAX_RETRIES):
        try:
            response = requests.post(
                API_URL,
                headers=headers,
                json=payload,
                timeout=REQUEST_TIMEOUT_S,
            )

            if response.status_code == 429:
                wait = BACKOFF_BASE * (2 ** attempt)
                tqdm.write(f"[RATE LIMIT] Waiting {wait:.1f}s before retrying.")
                time.sleep(wait)
                continue

            response.raise_for_status()
            body = response.json()
            return body["choices"][0]["message"]["content"]

        except requests.exceptions.RequestException as e:
            if attempt == MAX_RETRIES - 1:
                tqdm.write(f"[ERROR] {text[:60]!r} -> {e}")
                return None

            wait = BACKOFF_BASE * (2 ** attempt)
            tqdm.write(f"[RETRY] {e} | Waiting {wait:.1f}s before retrying.")
            time.sleep(wait)

    return None

### Run emotion predictions

In [ ]:
# Validate the required user inputs
if not API_KEY :
    raise ValueError("Set API_KEY before running the notebook.")

if not str(OUTPUT_NAME).strip():
    raise ValueError("Set OUTPUT_NAME before running the notebook.")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Run emotion predictions across the text column
eval_df = prep_df.copy()
raw_responses = []
predictions = []

for text in tqdm(eval_df[TEXT_COL], desc="Evaluating", unit="sample"):
    raw_response = call_llm(text)
    prediction = extract_label(raw_response)
    raw_responses.append(raw_response)
    predictions.append(prediction)
    time.sleep(REQUEST_GAP_S)

eval_df["raw_response"] = raw_responses
eval_df[PREDICT_COL] = predictions

successful = eval_df[PREDICT_COL].notna().sum()
print(f"Successful predictions: {successful}/{len(eval_df)}")
display(eval_df.head())

### Save summary JSON

In [ ]:
# Calculate overall and class-wise accuracy
scored_df = eval_df.dropna(subset=[PREDICT_COL]).copy()
if scored_df.empty:
    raise ValueError("No valid predictions were produced. Check the prompt format and API response.")

scored_df["is_correct"] = scored_df[EXPECTED_COL] == scored_df[PREDICT_COL]
overall_accuracy = round(float(scored_df["is_correct"].mean()), 2)

breakdown = (
    scored_df.groupby(EXPECTED_COL, sort=False)["is_correct"]
    .agg(correct="sum", total="size")
    .reset_index()
    .rename(columns={EXPECTED_COL: "emotion"})
)
breakdown["accuracy"] = (breakdown["correct"] / breakdown["total"]).round(2)

classes = {}
for emotion in EMOTION_ORDER:
    match = breakdown.loc[breakdown["emotion"] == emotion, "accuracy"]
    if not match.empty:
        classes[emotion] = float(match.iloc[0])

results = {
    "overall": overall_accuracy,
    "classes": classes,
}

output_name = str(OUTPUT_NAME).strip()
if not output_name.lower().endswith(".json"):
    output_name = f"{output_name}.json"

output_path = RESULTS_DIR / output_name
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Saved results to {output_path}")
display(breakdown)
results

Save reults in  `llm_results` folder, with one `overall` score and one `classes` object containing per-emotion accuracy values.